[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/templates/b_30_gqa_pure.ipynb)

# 🔴 Hard: Grouped-Query Attention without Flax

*Attention & Transformers*
Problem 10's grouped-query attention with no Flax.

### Signature
```python
class GroupQueryAttention:
    def __init__(self, d_model, num_heads, num_kv_heads, *, key): ...
    def __call__(self, x): ...        # (B, seq, d_model) -> (B, seq, d_model)
```

`num_heads` must be divisible by `num_kv_heads`. Set `self.d_k = d_model //
num_heads`.

### The projections are deliberately asymmetric
| | shape |
|---|---|
| `self.W_q` | `(d_model, d_model)` |
| `self.W_k` | `(d_model, num_kv_heads * d_k)` — **smaller** |
| `self.W_v` | `(d_model, num_kv_heads * d_k)` — **smaller** |
| `self.W_o` | `(d_model, d_model)` |

Two endpoints fall out of the same code:

- `num_kv_heads == num_heads` → ordinary multi-head attention
- `num_kv_heads == 1` → multi-query attention (MQA)

### Why it exists — the KV cache, not the FLOPs
Cache size is proportional to $H_{kv} \times seq \times d_k$, **not**
$H_q$. Dropping `num_kv_heads` from 64 to 8 makes the cache eight times
smaller and the per-step memory traffic eight times lighter, while the query
heads stay at 64 so quality barely moves. Llama-3-70B ships `H=64`, `H_{kv}=8`.

### `repeat`, not `tile`
After splitting, `q` has `num_heads` on the head axis and `k`/`v` have
`num_kv_heads`. Expand the KV heads to match:

```python
r = num_heads // num_kv_heads
k = jnp.repeat(k, r, axis=-3)
```

The two candidates group differently, and only one is right:

```
jnp.repeat([0,1,2,3], 2)  ->  [0,0,1,1,2,2,3,3]   ✅ adjacent query heads share a KV head
jnp.tile([0,1,2,3], 2)    ->  [0,1,2,3,0,1,2,3]   ❌ interleaved groups
```

Both give an array of the right shape, so a shape check will not catch it.

### Why this exists alongside problem 10
Interview sandboxes ship `jax` but not `flax`. Same class name, same argument
names, same `W_q`/`W_k`/`W_v`/`W_o` attributes — only
`rngs=nnx.Rngs(params=0)` becomes `key=jax.random.key(0)`.

In [ ]:
# Colab setup (no-op when running locally).
# jax-judge is not published on PyPI, so the judge is installed from the
# repo itself. Regenerate with JAXCODE_REPO=you/YourFork to point this at
# your own fork:  JAXCODE_REPO=you/JAXCode make notebooks
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q flax optax')
    get_ipython().run_line_magic(
        'pip', 'install -q git+https://github.com/YOUR-GITHUB-USERNAME/JAXCode.git')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp
from flax import nnx

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

import jax
import jax.numpy as jnp


class Linear:
    """Given to you, exactly as nnx.Linear is given to you in problem 10."""

    def __init__(self, d_in, d_out, *, key):
        self.kernel = jax.random.normal(key, (d_in, d_out)) / jnp.sqrt(d_in)
        self.bias = jnp.zeros((d_out,))

    def __call__(self, x):
        return x @ self.kernel + self.bias


class GroupQueryAttention:
    """num_heads query heads sharing num_kv_heads key/value heads."""

    def __init__(self, d_model, num_heads, num_kv_heads, *, key):
        pass  # Replace this

    def __call__(self, x):
        pass  # Replace this

In [ ]:
# 🔍 Scratch cell — poke at your implementation
import jax
import jax.numpy as jnp

for kvh in (4, 2, 1):
    g = GroupQueryAttention(16, num_heads=4, num_kv_heads=kvh, key=jax.random.key(0))
    label = {4: "= MHA", 2: "= GQA", 1: "= MQA"}[kvh]
    print(f"num_kv_heads={kvh} {label:<6} W_q {g.W_q.kernel.shape}  W_k {g.W_k.kernel.shape}")

x = jax.random.normal(jax.random.key(1), (2, 6, 16))
print("\nout:", GroupQueryAttention(16, 4, 2, key=jax.random.key(0))(x).shape)

print("\nrepeat vs tile, the grouping that matters:")
a = jnp.arange(4)
print("  repeat:", jnp.repeat(a, 2).tolist())
print("  tile:  ", jnp.tile(a, 2).tolist())

In [ ]:
# ✅ SUBMIT — run this cell to check your solution
from jax_judge import check, hint, solution, status

check("gqa_pure")

# hint("gqa_pure")      # stuck? nudge without the answer
# solution("gqa_pure")  # spoiler: the reference implementation
# status()              # your dashboard across all problems